In [ ]:
# !pip install pymodbus

In [7]:
from pymodbus.client import ModbusTcpClient
import time as tt
import threading

client = ModbusTcpClient('172.30.1.84', port=502)
client.connect()

True

In [8]:
# 물품 생산 - 가공 - 분류
from collections import deque


class FactoryController:
    SLAVE_ID = 1

    SCAN_INTERVAL = 0.03
    EMITTER_PULSE = 0.20
    PUSHER_PULSE = 0.80
    RESET_TIME = 0.50
    START_DELAY = 0.50
    STOP_TIME = 0.50

    CONVEYORS = tuple(range(0, 21)) + tuple(range(49, 62))

    PRODUCTION_LINES = (
        {
            "name": "Blue Product Lid",
            "sensor": 5,
            "emitter": 21,
        },
        {
            "name": "Blue Product Base",
            "sensor": 4,
            "emitter": 22,
        },
        {
            "name": "Green Product Lid",
            "sensor": 6,
            "emitter": 35,
        },
        {
            "name": "Green Product Base",
            "sensor": 7,
            "emitter": 36,
        },
    )

    MACHINE_MODES = {
        23: True,
        27: False,
        37: True,
        41: False,
    }

    MACHINE_STARTS = (
        24,
        28,
        38,
        42,
    )

    MACHINE_STOPS = (
        25,
        29,
        39,
        43,
    )

    MACHINE_RESETS = (
        26,
        30,
        40,
        44,
    )

    VISION_VALUES = {
        2: "Blue Product Lid",
        3: "Blue Product Base",
        5: "Green Product Lid",
        6: "Green Product Base",
    }

    SORT_STATIONS = (
        {
            "name": "Blue Product Base",
            "sensor": 1,
            "pusher": 31,
        },
        {
            "name": "Blue Product Lid",
            "sensor": 3,
            "pusher": 32,
        },
        {
            "name": "Green Product Lid",
            "sensor": 2,
            "pusher": 34,
        },
        {
            "name": "Green Product Base",
            "sensor": 0,
            "pusher": 33,
        },
    )

    STOP_STATIONS = (
        {
            "name": "Blue Product Base",
            "sensor": 8,
            "stop": 45,
        },
        {
            "name": "Blue Product Lid",
            "sensor": 9,
            "stop": 46,
        },
        {
            "name": "Green Product Lid",
            "sensor": 10,
            "stop": 47,
        },
        {
            "name": "Green Product Base",
            "sensor": 11,
            "stop": 48,
        },
    )

    def __init__(self, client):
        self.client = client
        self.stop_event = threading.Event()
        self.thread = None
        self.io_lock = threading.Lock()

        self.pulse_deadlines = {}
        self.sort_queues = [deque() for _ in range(4)]

        self.prev_inputs = [False] * 12
        self.prev_vision_active = False

        self.roller_latched = [False] * 4

        emitters = tuple(
            line["emitter"]
            for line in self.PRODUCTION_LINES
        )

        pushers = tuple(
            station["pusher"]
            for station in self.SORT_STATIONS
        )

        roller_stops = tuple(
            station["stop"]
            for station in self.STOP_STATIONS
        )

        self.managed_coils = tuple(
            sorted(
                set(
                    self.CONVEYORS
                    + emitters
                    + tuple(self.MACHINE_MODES.keys())
                    + self.MACHINE_STARTS
                    + self.MACHINE_STOPS
                    + self.MACHINE_RESETS
                    + pushers
                    + roller_stops
                )
            )
        )

    def _call(self, method, *args, **kwargs):
        last_error = None

        for key in ("device_id", "slave", "unit"):
            try:
                with self.io_lock:
                    return method(
                        *args,
                        **kwargs,
                        **{key: self.SLAVE_ID},
                    )
            except TypeError as error:
                last_error = error

        try:
            with self.io_lock:
                return method(*args, **kwargs)
        except TypeError:
            raise last_error

    @staticmethod
    def _response_ok(response):
        if response is None:
            return False

        checker = getattr(response, "isError", None)

        if callable(checker):
            return not checker()

        return True

    def _write(self, address, value):
        response = self._call(
            self.client.write_coil,
            address,
            bool(value),
        )

        if not self._response_ok(response):
            raise RuntimeError(
                f"Coil {address} 쓰기 실패"
            )

    def _safe_write(self, address, value):
        try:
            self._write(address, value)
        except Exception:
            pass

    def _read_inputs(self):
        response = self._call(
            self.client.read_discrete_inputs,
            0,
            count=12,
        )

        if not self._response_ok(response):
            return None

        bits = list(response.bits[:12])

        if len(bits) < 12:
            bits.extend(
                [False] * (12 - len(bits))
            )

        return [
            bool(value)
            for value in bits
        ]

    def _read_vision(self):
        response = self._call(
            self.client.read_input_registers,
            0,
            count=1,
        )

        if not self._response_ok(response):
            return None

        if not response.registers:
            return None

        return int(response.registers[0])

    def _pulse(self, address, duration):
        self._write(address, True)

        self.pulse_deadlines[address] = (
            tt.monotonic() + duration
        )

    def _update_pulses(self):
        now = tt.monotonic()

        finished = [
            address
            for address, deadline
            in self.pulse_deadlines.items()
            if now >= deadline
        ]

        for address in finished:
            self._write(address, False)
            self.pulse_deadlines.pop(
                address,
                None,
            )

    def _configure_outputs(self):
        for address in self.managed_coils:
            self._write(address, False)

        for address in self.CONVEYORS:
            self._write(address, True)

        for address, value in self.MACHINE_MODES.items():
            self._write(address, value)

        for address in self.MACHINE_STOPS:
            self._write(address, False)

        for address in self.MACHINE_STARTS:
            self._write(address, False)

        for address in self.MACHINE_RESETS:
            self._write(address, True)

        tt.sleep(self.RESET_TIME)

        for address in self.MACHINE_RESETS:
            self._write(address, False)

        tt.sleep(self.START_DELAY)

        for address in self.MACHINE_STARTS:
            self._write(address, True)

        print(
            "[가공 시작] Machining Center 0, 1, 2, 3"
        )

        tt.sleep(self.START_DELAY)

        for line in self.PRODUCTION_LINES:
            self._pulse(
                line["emitter"],
                self.EMITTER_PULSE,
            )

            print(
                f'[생성] {line["name"]} 원자재 투입'
            )

    def _process_production(self, inputs):
        for line in self.PRODUCTION_LINES:
            sensor = line["sensor"]

            rising = (
                inputs[sensor]
                and not self.prev_inputs[sensor]
            )

            if not rising:
                continue

            print(
                f'[가공 완료] {line["name"]}'
            )

            self._pulse(
                line["emitter"],
                self.EMITTER_PULSE,
            )

            print(
                f'[생성] {line["name"]} 다음 원자재 투입'
            )

    def _process_vision(self, value):
        active = value != 0

        if active and not self.prev_vision_active:
            product = self.VISION_VALUES.get(value)

            if product is None:
                print(
                    f"[검증 실패] Vision Sensor 값: {value}"
                )
            else:
                self.sort_queues[0].append(product)

                print(
                    f"[검증 완료] {product}"
                )

        self.prev_vision_active = active

    def _process_sorting(self, inputs):
        for index, station in enumerate(
            self.SORT_STATIONS
        ):
            sensor = station["sensor"]

            rising = (
                inputs[sensor]
                and not self.prev_inputs[sensor]
            )

            if not rising:
                continue

            if not self.sort_queues[index]:
                print(
                    f'[분류 오류] {station["name"]} 위치의 '
                    f"추적 물품 없음"
                )
                continue

            product = self.sort_queues[index].popleft()

            if product == station["name"]:
                self._pulse(
                    station["pusher"],
                    self.PUSHER_PULSE,
                )

                print(
                    f"[분류 완료] {product}"
                )

            elif index < len(self.SORT_STATIONS) - 1:
                self.sort_queues[index + 1].append(
                    product
                )

            else:
                print(
                    f"[분류 실패] {product}"
                )

    def _process_roller_stops(self, inputs):
        for index, station in enumerate(
            self.STOP_STATIONS
        ):
            sensor = station["sensor"]

            rising = (
                inputs[sensor]
                and not self.prev_inputs[sensor]
            )

            if not rising:
                continue

            if self.roller_latched[index]:
                continue

            self._write(
                station["stop"],
                True,
            )

            self.roller_latched[index] = True

            print(
                f'[적재 대기] {station["name"]} - '
                f"Roller Stop {index} 작동"
            )

    def _shutdown_outputs(self):
        self.pulse_deadlines.clear()

        for address in self.MACHINE_STARTS:
            self._safe_write(address, False)

        for address in self.MACHINE_STOPS:
            self._safe_write(address, True)

        tt.sleep(self.STOP_TIME)

        for address in self.MACHINE_STOPS:
            self._safe_write(address, False)

        for address in self.managed_coils:
            self._safe_write(address, False)

    def _run(self):
        try:
            while not self.stop_event.is_set():
                self._update_pulses()

                inputs = self._read_inputs()
                vision_value = self._read_vision()

                if inputs is None or vision_value is None:
                    print(
                        "[통신 오류] Factory I/O 입력 읽기 실패"
                    )

                    self.stop_event.wait(
                        self.SCAN_INTERVAL
                    )
                    continue

                self._process_production(inputs)
                self._process_vision(vision_value)
                self._process_sorting(inputs)
                self._process_roller_stops(inputs)

                self.prev_inputs = inputs

                self.stop_event.wait(
                    self.SCAN_INTERVAL
                )

        except Exception as error:
            print(
                f"[제어 오류] {error}"
            )

            self.stop_event.set()

        finally:
            self._shutdown_outputs()

    def start(self):
        if (
            self.thread is not None
            and self.thread.is_alive()
        ):
            return

        self.stop_event.clear()
        self.pulse_deadlines.clear()

        self.sort_queues = [
            deque()
            for _ in range(4)
        ]

        self.prev_inputs = [False] * 12
        self.prev_vision_active = False
        self.roller_latched = [False] * 4

        self._configure_outputs()

        self.thread = threading.Thread(
            target=self._run,
            name="factory-control",
            daemon=True,
        )

        self.thread.start()

        print(
            "[제어 시작] 생성 → 가공 → 검증 → 분류 → 적재 대기"
        )

    def stop(self):
        self.stop_event.set()

        if (
            self.thread is not None
            and self.thread.is_alive()
        ):
            self.thread.join(timeout=5)

        if (
            self.thread is not None
            and self.thread.is_alive()
        ):
            self._shutdown_outputs()

        print(
            "[제어 종료] 전체 설비 정지"
        )

In [9]:
# 조립 제어 코드
class AssemblyController:
    SCAN_INTERVAL = 0.03

    CLAMP_TIME = 0.8
    Z_MOVE_TIME = 0.9
    X_MOVE_TIME = 1.2
    GRAB_TIME = 0.4
    RAISE_TIME = 1.2
    EMITTER_PULSE = 0.2

    EXIT_SENSOR_START = 12

    LINES = (
        {
            "name": "Blue Product",
            "ready_indices": (0, 1),
            "clamps": (65, 66),
            "raise": 69,
            "x": 71,
            "z": 72,
            "grab": 73,
            "exit_index": 0,
            "stops": (45, 46),
            "emitter": 77,
            "pallet": "Pallet 1",
        },
        {
            "name": "Green Product",
            "ready_indices": (2, 3),
            "clamps": (67, 68),
            "raise": 70,
            "x": 74,
            "z": 75,
            "grab": 76,
            "exit_index": 1,
            "stops": (47, 48),
            "emitter": 78,
            "pallet": "Pallet 0",
        },
    )

    def __init__(self, factory_controller):
        self.factory = factory_controller

        self.stop_event = threading.Event()
        self.thread = None

        self.pulse_deadlines = {}
        self.prev_exit_inputs = [False, False]

        self.states = [
            {
                "stage": "WAIT_PARTS",
                "deadline": 0.0,
            }
            for _ in self.LINES
        ]

        addresses = []

        for line in self.LINES:
            addresses.extend(line["clamps"])
            addresses.extend(
                (
                    line["raise"],
                    line["x"],
                    line["z"],
                    line["grab"],
                    line["emitter"],
                )
            )

        self.managed_coils = tuple(
            sorted(set(addresses))
        )

    def _write(self, address, value):
        self.factory._write(
            address,
            value,
        )

    def _safe_write(self, address, value):
        self.factory._safe_write(
            address,
            value,
        )

    def _read_exit_inputs(self):
        response = self.factory._call(
            self.factory.client.read_discrete_inputs,
            self.EXIT_SENSOR_START,
            count=2,
        )

        if not self.factory._response_ok(response):
            return None

        bits = list(response.bits[:2])

        if len(bits) < 2:
            bits.extend(
                [False] * (2 - len(bits))
            )

        return [
            bool(value)
            for value in bits
        ]

    def _pulse(self, address, duration):
        self._write(address, True)

        self.pulse_deadlines[address] = (
            tt.monotonic() + duration
        )

    def _update_pulses(self):
        now = tt.monotonic()

        finished = [
            address
            for address, deadline
            in self.pulse_deadlines.items()
            if now >= deadline
        ]

        for address in finished:
            self._write(address, False)

            self.pulse_deadlines.pop(
                address,
                None,
            )

    def _parts_ready(self, line):
        return all(
            self.factory.roller_latched[index]
            for index in line["ready_indices"]
        )

    def _process_line(self, index, now):
        line = self.LINES[index]
        state = self.states[index]
        stage = state["stage"]

        if stage == "WAIT_PARTS":
            if not self._parts_ready(line):
                return

            for address in line["clamps"]:
                self._write(address, True)

            state["stage"] = "CLAMPING"
            state["deadline"] = now + self.CLAMP_TIME

            print(
                f'[조립 시작] {line["name"]}'
            )

            return

        if stage == "WAIT_EXIT":
            return

        if now < state["deadline"]:
            return

        if stage == "CLAMPING":
            for address in line["clamps"]:
                self._write(address, False)

            self._write(line["z"], True)

            state["stage"] = "PICK_DOWN"
            state["deadline"] = now + self.Z_MOVE_TIME

        elif stage == "PICK_DOWN":
            self._write(line["grab"], True)

            state["stage"] = "GRABBING"
            state["deadline"] = now + self.GRAB_TIME

        elif stage == "GRABBING":
            self._write(line["z"], False)

            state["stage"] = "PICK_UP"
            state["deadline"] = now + self.Z_MOVE_TIME

        elif stage == "PICK_UP":
            self._write(line["x"], False)

            state["stage"] = "MOVE_TO_BASE"
            state["deadline"] = now + self.X_MOVE_TIME

        elif stage == "MOVE_TO_BASE":
            self._write(line["z"], True)

            state["stage"] = "PLACE_DOWN"
            state["deadline"] = now + self.Z_MOVE_TIME

        elif stage == "PLACE_DOWN":
            self._write(line["grab"], False)

            state["stage"] = "RELEASING"
            state["deadline"] = now + self.GRAB_TIME

        elif stage == "RELEASING":
            self._write(line["z"], False)

            state["stage"] = "PLACE_UP"
            state["deadline"] = now + self.Z_MOVE_TIME

        elif stage == "PLACE_UP":
            self._write(line["raise"], True)

            state["stage"] = "PASSING"
            state["deadline"] = now + self.RAISE_TIME

        elif stage == "PASSING":
            self._write(line["raise"], False)
            self._write(line["x"], True)

            state["stage"] = "RETURNING"
            state["deadline"] = now + self.X_MOVE_TIME

        elif stage == "RETURNING":
            state["stage"] = "WAIT_EXIT"

            print(
                f'[조립 완료] {line["name"]}'
            )

    def _process_exit_inputs(self, inputs):
        for index, line in enumerate(self.LINES):
            sensor_index = line["exit_index"]

            rising = (
                inputs[sensor_index]
                and not self.prev_exit_inputs[sensor_index]
            )

            if not rising:
                continue

            if self.states[index]["stage"] != "WAIT_EXIT":
                continue

            for address in line["stops"]:
                self._write(address, False)

            for latch_index in line["ready_indices"]:
                self.factory.roller_latched[
                    latch_index
                ] = False

            self._pulse(
                line["emitter"],
                self.EMITTER_PULSE,
            )

            self.states[index] = {
                "stage": "WAIT_PARTS",
                "deadline": 0.0,
            }

            print(
                f'[배출 완료] {line["name"]} → '
                f'{line["pallet"]} 생성'
            )

        self.prev_exit_inputs = inputs

    def _shutdown_outputs(self):
        self.pulse_deadlines.clear()

        for address in self.managed_coils:
            self._safe_write(address, False)

    def _run(self):
        try:
            while not self.stop_event.is_set():
                self._update_pulses()

                exit_inputs = self._read_exit_inputs()

                if exit_inputs is not None:
                    self._process_exit_inputs(
                        exit_inputs
                    )

                now = tt.monotonic()

                for index in range(len(self.LINES)):
                    self._process_line(
                        index,
                        now,
                    )

                self.stop_event.wait(
                    self.SCAN_INTERVAL
                )

        except Exception as error:
            print(
                f"[조립 오류] {error}"
            )

            self.stop_event.set()

        finally:
            self._shutdown_outputs()

    def start(self):
        if (
            self.thread is not None
            and self.thread.is_alive()
        ):
            return

        self.stop_event.clear()
        self.pulse_deadlines.clear()

        self.states = [
            {
                "stage": "WAIT_PARTS",
                "deadline": 0.0,
            }
            for _ in self.LINES
        ]

        for address in self.managed_coils:
            self._write(address, False)

        for line in self.LINES:
            self._write(line["x"], True)

        exit_inputs = self._read_exit_inputs()

        if exit_inputs is not None:
            self.prev_exit_inputs = exit_inputs
        else:
            self.prev_exit_inputs = [False, False]

        self.thread = threading.Thread(
            target=self._run,
            name="assembly-control",
            daemon=True,
        )

        self.thread.start()

        print(
            "[조립 제어 시작] 부품 정렬 → 조립 → 배출"
        )

    def stop(self):
        self.stop_event.set()

        if (
            self.thread is not None
            and self.thread.is_alive()
        ):
            self.thread.join(timeout=5)

        if (
            self.thread is not None
            and self.thread.is_alive()
        ):
            self._shutdown_outputs()

        print(
            "[조립 제어 종료]"
        )

In [ ]:
if "assembly_controller" in globals():
    assembly_controller.stop()

if "factory_controller" in globals():
    factory_controller.stop()

factory_controller = FactoryController(client)
factory_controller.start()

assembly_controller = AssemblyController(
    factory_controller
)
assembly_controller.start()

[조립 제어 종료]
[제어 종료] 전체 설비 정지
[가공 시작] Machining Center 0, 1, 2, 3
[생성] Blue Product Lid 원자재 투입
[생성] Blue Product Base 원자재 투입
[생성] Green Product Lid 원자재 투입
[생성] Green Product Base 원자재 투입
[제어 시작] 생성 → 가공 → 검증 → 분류 → 적재 대기
[조립 제어 시작] 부품 정렬 → 조립 → 배출


[가공 완료] Blue Product Lid
[생성] Blue Product Lid 다음 원자재 투입
[가공 완료] Green Product Base
[생성] Green Product Base 다음 원자재 투입
[가공 완료] Blue Product Base
[생성] Blue Product Base 다음 원자재 투입
[가공 완료] Green Product Lid
[생성] Green Product Lid 다음 원자재 투입
[검증 완료] Green Product Base
[검증 완료] Blue Product Base
[분류 완료] Blue Product Base
[분류 완료] Green Product Base
[적재 대기] Blue Product Base - Roller Stop 0 작동
[적재 대기] Green Product Base - Roller Stop 3 작동
[검증 완료] Blue Product Lid
[가공 완료] Green Product Base
[생성] Green Product Base 다음 원자재 투입
[가공 완료] Blue Product Lid
[생성] Blue Product Lid 다음 원자재 투입
[가공 완료] Blue Product Base
[생성] Blue Product Base 다음 원자재 투입
[가공 완료] Green Product Lid
[생성] Green Product Lid 다음 원자재 투입
[적재 대기] Blue Product Lid - Roller Stop 1 작동
[조립 시작] Blue Product
[검증 완료] Green Product Base
[분류 완료] Blue Product Lid
[검증 완료] Blue Product Base
[분류 완료] Blue Product Base
[검증 완료] Green Product Lid
[조립 완료] Blue Product
[분류 완료] Green Product Base
[검증 완료] Blue Product Lid
[분류 완료] Green Product Lid
[가공 완료] Gree

Connection to (172.30.1.84, 502) failed: [Errno 111] Connection refused
Repeating....


In [ ]:
if "assembly_controller" in globals():
    assembly_controller.stop()

if "factory_controller" in globals():
    factory_controller.stop()

[제어 종료] 전체 설비 정지
